# 🥈 Silver Layer — Development Notebook

Use this notebook to test and develop the Silver layer logic locally.

**Flow:** Bronze data → Cleansing, dedup, quality filters → silver_stock_data

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SilverLayer_Dev")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Load Bronze Output
Read from the local parquet saved by the Bronze notebook, or create sample data.

In [ ]:
import os

if os.path.exists("data/test/bronze_output"):
    bronze_df = spark.read.parquet("data/test/bronze_output")
    print(f"Loaded Bronze output: {bronze_df.count()} rows")
else:
    # Fallback: create sample data
    sample_data = [
        ("AAPL", "2024-01-02", 185.5, 187.2, 184.8, 186.9, 186.9, 50000000, 0.0, 1.0, "file1.csv"),
        ("AAPL", "2024-01-03", 186.5, 188.0, 185.0, 185.5, 185.5, 48000000, 0.0, 1.0, "file1.csv"),
        ("AAPL", "2024-01-04", 182.0, 183.5, 181.0, 182.7, 182.7, 55000000, 0.0, 1.0, "file1.csv"),
        ("AAPL", "2024-01-02", 185.5, 187.2, 184.8, 186.9, 186.9, 50000000, 0.0, 1.0, "file2.csv"),  # duplicate
        ("GOOGL", "2024-01-02", 140.5, 142.0, 139.8, 141.2, 141.2, 30000000, 0.0, 1.0, "file1.csv"),
        (None,    "2024-01-02", 100.0, 101.0, 99.0,  100.5, 100.5, 10000000, 0.0, 1.0, "file1.csv"),  # null symbol
        ("MSFT",  None,         100.0, 101.0, 99.0,  100.5, 100.5, 10000000, 0.0, 1.0, "file1.csv"),  # null date
        ("MSFT", "2024-01-02", -5.0,  101.0, 99.0,  100.5, 100.5, 10000000, 0.0, 1.0, "file1.csv"),  # negative open
    ]
    columns = ["symbol", "date", "open", "high", "low", "close", "adjusted_close", "volume", "dividend_amount", "split_coefficient", "source_file"]
    bronze_df = spark.createDataFrame(sample_data, columns)
    bronze_df = bronze_df.withColumn("load_timestamp", F.current_timestamp()).withColumn("load_date", F.current_date())
    print(f"Created sample Bronze data: {bronze_df.count()} rows")

bronze_df.show(truncate=False)

## 2. Silver Logic — Step by Step

In [ ]:
bronze_count = bronze_df.count()
print(f"Starting Bronze count: {bronze_count}")

# Step 1: Cast date
df = bronze_df.withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))

# Step 2: Remove null critical fields
df = df.filter(
    F.col("symbol").isNotNull() &
    F.col("date").isNotNull() &
    F.col("close").isNotNull()
)
null_dropped = bronze_count - df.count()
print(f"Rows dropped (null critical fields): {null_dropped}")

In [ ]:
# Step 3: Quality filters
before_quality = df.count()
df = df.filter(
    (F.col("open") > 0) &
    (F.col("high") > 0) &
    (F.col("low") > 0) &
    (F.col("close") > 0) &
    (F.col("volume") >= 0)
)

df = df.filter(
    (F.col("high") >= F.col("low")) &
    (F.col("high") >= F.col("open")) &
    (F.col("high") >= F.col("close")) &
    (F.col("low") <= F.col("open")) &
    (F.col("low") <= F.col("close"))
)
quality_dropped = before_quality - df.count()
print(f"Rows dropped (quality filters): {quality_dropped}")
df.show(truncate=False)

In [ ]:
# Step 4: Deduplication
before_dedup = df.count()
w = Window.partitionBy("symbol", "date").orderBy(F.col("load_timestamp").desc())
df = (
    df
    .withColumn("_row_num", F.row_number().over(w))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)
dedup_dropped = before_dedup - df.count()
print(f"Duplicates removed: {dedup_dropped}")
print(f"Rows after dedup:   {df.count()}")

In [ ]:
# Step 5: Fill nulls
df = df.fillna({"adjusted_close": 0.0, "dividend_amount": 0.0, "split_coefficient": 1.0})

# Step 6: Add silver metadata
silver_df = df.withColumn("silver_load_timestamp", F.current_timestamp())

# Final output
silver_df = silver_df.select(
    "symbol", "date", "open", "high", "low", "close", "adjusted_close",
    "volume", "dividend_amount", "split_coefficient",
    "source_file", "load_timestamp", "silver_load_timestamp"
)

print(f"\n📊 Silver Data Quality Summary:")
print(f"   Bronze input:     {bronze_count}")
print(f"   Nulls removed:    {null_dropped}")
print(f"   Quality filtered: {quality_dropped}")
print(f"   Deduped:          {dedup_dropped}")
print(f"   Silver output:    {silver_df.count()}")

silver_df.show(truncate=False)

In [ ]:
# Save for Gold notebook
silver_df.write.mode("overwrite").parquet("data/test/silver_output")
print("✅ Silver output saved to data/test/silver_output/")

In [ ]:
# spark.stop()